In [ ]:
#settup av data
import hyperspy.api as hs
import pyxem as pxm
import matplotlib.pyplot as plt
from skimage import data
from skimage.filters import threshold_otsu
s = hs.load("006.zspy", lazy=True)
s_bs = s.get_direct_beam_position(method="center_of_mass")
s_bs.compute()
s_bs_lp = s_bs.get_linear_plane()
s_bs_corr = s_bs-s_bs_lp
s_sum = s.sum(axis=(-1, -2))
s_sum.compute()

In [ ]:
import numpy as np
import scipy.ndimage as nd
from scipy.optimize import root
from scipy.interpolate import RegularGridInterpolator
from skimage.filters import sobel


def pinhole(sh, r=0.8):
    x = np.linspace(-1, 1, num=sh[0])
    y = np.linspace(-1, 1, num=sh[1])
    xx, yy = np.meshgrid(x, y, indexing='ij')
    return xx**2 + yy**2 < r**2
def normalize(img, axis=None):
    if axis is None:
        axis = (-2, -1)
    if np.iscomplexobj(img):
        raise TypeError(
            "You don't want to divide with maximum for complex dtypes here."
        )
    img = img - img.min(axis=axis, keepdims=True)
    img = img / img.max(axis=axis, keepdims=True)
    return img